<a href="https://colab.research.google.com/github/hafizapatel04-bit/f1_Prediction/blob/main/predictionOfBrizilliangp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install fastf1 lightgbm pandas numpy scikit-learn


In [ ]:
import fastf1
import pandas as pd
from tqdm import tqdm  # For progress bar

# Enable cache
fastf1.Cache.enable_cache('/content/f1_cache')

def load_season_data(years):
    """Load F1 race results for specified years."""
    results = []

    for year in years:
        print(f"\n{'='*50}")
        print(f"Processing {year} season")
        print('='*50)

        try:
            # Get event schedule
            cal = fastf1.get_event_schedule(year)
            races = cal[cal['EventFormat'] == 'conventional']
            print(f"Found {len(races)} conventional races")

            # Load each race
            for idx, race_name in enumerate(races['EventName'], 1):
                try:
                    print(f"[{idx}/{len(races)}] Loading {race_name}...", end=' ')

                    session = fastf1.get_session(year, race_name, 'R')
                    session.load()

                    # Extract relevant columns
                    df = session.results[['Abbreviation', 'TeamName', 'Position']].copy()
                    df['Year'] = year
                    df['Race'] = race_name

                    results.append(df)
                    print("✓")

                except Exception as e:
                    print(f"✗ ({str(e)[:50]})")

        except Exception as e:
            print(f"✗ Failed to load {year} schedule: {e}")

    return results

# Load data for desired years
years = [2023]  # Add more years like [2021, 2022, 2023]
results = load_season_data(years)

# Combine and clean data
if results:
    data = pd.concat(results, ignore_index=True)
    data['Position'] = pd.to_numeric(data['Position'], errors='coerce')

    print(f"\n{'='*50}")
    print(f"✓ Successfully loaded {len(results)} races")
    print(f"  Total entries: {len(data)}")
    print(f"  Years: {sorted(data['Year'].unique())}")
    print(f"  Races: {data['Race'].nunique()}")
    print('='*50)

    display(data.head(10))
else:
    print("\n✗ No data was loaded")

logger      WARNING 	Failed to load schedule from FastF1 backend!
DEBUG:fastf1.fastf1.events:Traceback for failure in FastF1 schedule
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/fastf1/logger.py", line 151, in __wrapped
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastf1/events.py", line 599, in _get_schedule_ff1
    response = Cache.requests_get(
               ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastf1/req.py", line 303, in requests_get
    return cls._cached_request('GET', url, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastf1/req.py", line 347, in _cached_request
    response = func(url, **kwargs)
               ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/requests_cache/session.py", line 127, in get
    return self.request('GET', url, params=params, **kw


Processing 2023 season


logger      WARNING 	Failed to load schedule from F1 API backend!
DEBUG:fastf1.fastf1.events:Traceback for failure in F1 API schedule
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/fastf1/logger.py", line 151, in __wrapped
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastf1/events.py", line 659, in _get_schedule_from_f1_timing
    response = fastf1._api.season_schedule(f'/static/{year}/')
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastf1/req.py", line 479, in _cached_api_request
    data = func(api_path, **func_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastf1/_api.py", line 1678, in season_schedule
    response = fetch_page(path, 'index')
               ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastf1/_api.py", line 1752, in fetch

✗ Failed to load 2023 schedule: Failed to load any schedule data.

✗ No data was loaded


In [ ]:
# Calculate Driver and Team Form (last 5 races average position)

# Driver Form
data['DriverForm'] = (
    data.groupby('Abbreviation')['Position']
        .rolling(5, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
)

# Team Form
data['TeamForm'] = (
    data.groupby('TeamName')['Position']
        .rolling(5, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
)



In [ ]:
data['Wins'] = (data['Position'] == 1).astype(int)


In [ ]:
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

# ============================================
# STEP 1: Feature Engineering
# ============================================

def calculate_form(data, window=5):
    """Calculate rolling form for drivers and teams."""
    data = data.sort_values(['Year', 'Race']).copy()
    data['win'] = (data['Position'] == 1).astype(int)

    # Calculate average finishing position (lower is better)
    data['DriverForm'] = data.groupby('Abbreviation')['Position'].transform(
        lambda x: x.rolling(window=window, min_periods=1).mean()
    )

    data['TeamForm'] = data.groupby('TeamName')['Position'].transform(
        lambda x: x.rolling(window=window, min_periods=1).mean()
    )

    return data

# Apply feature engineering
data = calculate_form(data, window=5)

# Remove rows with missing values
data = data.dropna(subset=['DriverForm', 'TeamForm'])

print(f"Dataset shape: {data.shape}")
print(f"Win distribution:\n{data['win'].value_counts()}")

# ============================================
# STEP 2: Train Model
# ============================================

features = data[['DriverForm', 'TeamForm']]
target = data['win']

X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, shuffle=True, random_state=42, stratify=target
)

model = LGBMClassifier(
    num_leaves=31,
    learning_rate=0.05,
    n_estimators=100,
    random_state=42,
    verbose=-1
)

model.fit(X_train, y_train)

# ============================================
# STEP 3: Evaluate Model
# ============================================

print("\n" + "="*50)
print("MODEL PERFORMANCE")
print("="*50)
print(f"Training Accuracy: {model.score(X_train, y_train):.4f}")
print(f"Test Accuracy: {model.score(X_test, y_test):.4f}")

y_pred_proba = model.predict_proba(X_test)[:, 1]
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}")

print("\nFeature Importance:")
for feature, importance in zip(features.columns, model.feature_importances_):
    print(f"  {feature}: {importance:.4f}")

# ============================================
# STEP 4: Make Predictions
# ============================================

drivers = ["VER", "NOR", "PIA", "LEC", "HAM"]
driver_form = np.array([2.1, 2.4, 4.0, 3.5, 5.2])  # Lower is better
team_form = np.array([2.5, 2.2, 2.2, 3.0, 4.1])

X_pred = pd.DataFrame({
    'DriverForm': driver_form,
    'TeamForm': team_form
})

probs = model.predict_proba(X_pred)[:, 1]

predictions = pd.DataFrame({
    'Driver': drivers,
    'DriverForm': driver_form,
    'TeamForm': team_form,
    'Win_Probability': probs * 100  # Convert to percentage
}).sort_values(by='Win_Probability', ascending=False)

print("\n" + "="*50)
print("RACE WIN PREDICTIONS")
print("="*50)
predictions['Win_Probability'] = predictions['Win_Probability'].apply(lambda x: f"{x:.2f}%")
print(predictions.to_string(index=False))

Dataset shape: (59, 9)
Win distribution:
win
0    56
1     3
Name: count, dtype: int64

MODEL PERFORMANCE
Training Accuracy: 0.9574
Test Accuracy: 0.9167
ROC-AUC Score: 0.7727

Feature Importance:
  DriverForm: 76.0000
  TeamForm: 24.0000

RACE WIN PREDICTIONS
Driver  DriverForm  TeamForm Win_Probability
   VER         2.1       2.5           6.90%
   NOR         2.4       2.2           6.90%
   PIA         4.0       2.2           6.90%
   LEC         3.5       3.0           6.90%
   HAM         5.2       4.1           6.90%
